# Now let's fully train our network!

In [1]:
! pip install torchvision

In [2]:
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

X_train = train_dataset.data.numpy().astype(np.float32) / 255.0
y_train = train_dataset.targets.numpy()

X_test = test_dataset.data.numpy().astype(np.float32) / 255.0
y_test = test_dataset.targets.numpy()

X_train = X_train.reshape(-1, 784)
X_test = X_test.reshape(-1, 784)

In [3]:
from neural_net import MLP
from autograd import Value

model = MLP(784, [100, 50], 10, [Value.relu, Value.relu])

In [4]:
def softmax(x:list):
    x_max = float('-inf')
    for i in range(len(x)):
        x_max = max(x_max, x[i].data)
        
    # print(x_max)
    x_max = Value(x_max)
    
    output = []
    denom = 0
    #print(x)
    for i in range(len(x)):
        cur = (x[i] - x_max).exp()
        output += [cur]
        denom += cur
        
    for i in range(len(output)):
        output[i] = output[i] / denom
                
    return output

In [5]:
def cross_entropy_loss(p_target, p_pred):
    if not isinstance(p_target, list):
        p_list = [Value(0.0) for _ in range(len(p_pred))]
        p_list[p_target] = Value(1.)
        p_target = p_list
    
    loss = Value(0.0)
    for i in range(len(p_target)):
        loss -= p_target[i] * (p_pred[i]).log()
        
    return loss

In [16]:
def eval(model, test_X, test_Y):
    num_correct = 0
    for x, y in zip(test_X, test_Y):
        y_pred = model.forward(x)
        y_pred_probs = softmax(y_pred)
        
        max_prob = -1
        max_idx = 0
        for i in range(len(y_pred_probs)):
            if y_pred_probs[i].data > max_prob:
                max_prob = y_pred_probs[i].data
                max_idx = i
                
        if max_idx == y:
            num_correct += 1
            
    return num_correct / len(test_X)

In [ ]:
eval(model, X_train[:10], y_train[:10])

In [6]:
def train(model, train_X, train_Y, epochs, optimizer):
    for epoch in range(epochs):
        epoch_loss = []
        
        for x, y_target in zip(train_X[:5], train_Y[:5]):
            optim.zero_grad() # don't forget this step
            print("BEFORE FORWARD")
            y_pred = model.forward(x)
            
            y_pred_probs = softmax(y_pred)
            print("AFTER FORWARD")
            loss = cross_entropy_loss(y_target, y_pred_probs)
            epoch_loss += [loss]
            
            print(loss)
            
            loss.backward()
            optimizer.step()
            
        avg_loss = 0
        for i in range(len(epoch_loss)):
            avg_loss += epoch_loss[i]
            
        avg_loss = avg_loss / len(epoch_loss)
            
        print(f"Epoch {epoch} average loss: {avg_loss.data}")

In [11]:
class GradientDescent:
    def __init__(self, params, lr):
        self.params = params
        self.lr = lr
        
    def flatten_params(self, params):
        flat = []

        for p in params:
            if isinstance(p, list):
                flat.extend(self.flatten_params(p))
            else:
                flat.append(p)

        return flat
        
    def step(self):
        for param in self.flatten_params(self.params):
            param.data -= self.lr * param.grad
            
    def zero_grad(self):
        for param in self.flatten_params(self.params):
            param.grad = 0
    

optim = GradientDescent(model.parameters(), 1e-2)

In [12]:
train(model, X_train, y_train, 10, optim)

BEFORE FORWARD
AFTER FORWARD
:2.302388933876179
BEFORE FORWARD
AFTER FORWARD
:2.303942814237238
BEFORE FORWARD
AFTER FORWARD
:2.3051217323709383
BEFORE FORWARD
AFTER FORWARD
:2.305785035783712
BEFORE FORWARD
AFTER FORWARD
:2.3067874924494194
Epoch 0 average loss: 2.304805201743498
BEFORE FORWARD
AFTER FORWARD
:2.297354570239702
BEFORE FORWARD
AFTER FORWARD
:2.298923893879435
BEFORE FORWARD
AFTER FORWARD
:2.3001294715585754
BEFORE FORWARD
AFTER FORWARD
:2.300809509034609
BEFORE FORWARD
AFTER FORWARD
:2.3017856345917127
Epoch 1 average loss: 2.299800615860807
BEFORE FORWARD
AFTER FORWARD
:2.292372017711141
BEFORE FORWARD
AFTER FORWARD
:2.293955353258546
BEFORE FORWARD
AFTER FORWARD
:2.2951875906505372
BEFORE FORWARD
AFTER FORWARD
:2.295882663009326
BEFORE FORWARD
AFTER FORWARD
:2.296833575216268
Epoch 2 average loss: 2.294846239969164
BEFORE FORWARD
AFTER FORWARD
:2.2874383691055336
BEFORE FORWARD
AFTER FORWARD
:2.2890365135241013
BEFORE FORWARD
AFTER FORWARD
:2.290292739803316
BEFORE FO